In [ ]:
#!git clone https://github.com/whyhardt/SPICE.git

In [ ]:
# !pip install -e SPICE

In [5]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from spice import SpiceEstimator, SpiceConfig, csv_to_dataset, BaseModel, plot_session, split_data_along_sessiondim

sys.path.append('../../..')
from weinhardt2026.utils.benchmarking_gru import Model as GRU, training
from weinhardt2026.studies.braun2018.benchmarking_braun2018 import ExpectedValueControl

# For custom RNN
import torch
import torch.nn as nn

# NOTEBOOK CONFIG

In [48]:
train_spice = False
train_evc = False
train_gru = True

# DATASET

## Load dataset

Let's load the data first with the `convert_dataset` method. This method returns a `SpiceDataset` object which we can use right away 

In [51]:
# Load your data
path_file = '../data/braun2018/braun2018_2.csv'

# dataset description from github: 
# subject -- subject ID 
# rsi -- response-stimulus interval, how long points were visible before stimulus onset
# block -- position in the experiment (out of 12)
# block_c -- same as above except the distribution is centered at block 6 (total number of blocks = 12)
# blocktime -- amount of time it took to complete a given block
# trial -- each row represents one trial
# current -- factor var representing whether the value of the task performed on previous trial changed (1 = decrease by a point, 0 = remained constant)
# other -- same as ^ except represents value changes for task not performed on previous trial and (1 = increase by a point, 0 = remain constant)
# difference -- difference between values on present trial (other - current; pos values indicate difference in favor of other task)
# stim.rep -- did the stimulus repeat from the previous trial
# pvsum -- sum of points
# rt -- amount of time between stimulus onset and response on present trial
# transcode -- task selection at trial N relative to selection at trial N-1 (1 = switched to the opposite task, 0 = repeated same task)

dataset = csv_to_dataset(
    file = path_file,
    df_participant_id='subject',
    df_choice='transcode',
    df_feedback=None,
    df_block='block',
    additional_inputs=('difference', 'current', 'other'),
    timeshift_additional_inputs=(-1, -1, -1),
    )

n_participants = len(dataset.xs[..., -1].unique())
n_actions = dataset.ys.shape[-1]

print(f"Shape of dataset: {dataset.xs.shape}")
print(f"Number of participants: {n_participants}")
print(f"Number of actions in dataset: {n_actions}")
print(f"Number of additional inputs: {dataset.xs.shape[-1]-2*n_actions-5}")

Shape of dataset: torch.Size([756, 274, 1, 10])
Number of participants: 63
Number of actions in dataset: 2
Number of additional inputs: 1


## Dataset processing

In [52]:
# seems like there are a few blocks that are way longer than the average block
# maybe limiting the block length to a given size would help
n_trials_per_block = np.zeros(dataset.xs.shape[0])
for index, block in enumerate(dataset.xs[:, :, 0, 0]):
    n_trials_per_block[index] = block.shape[0]-block.isnan().sum()

mean, std = n_trials_per_block.mean(), n_trials_per_block.std()

print(f"Average sequence length: {int(mean)}+-{int(std)}")
print(f"Old max sequence length is {dataset.xs.shape[1]}")

# it would be okay to limit until ca mean+2*std trials per  (~200 trials)
dataset.xs = dataset.xs[:, :int(mean+2*std)]
dataset.ys = dataset.ys[:, :int(mean+2*std)]

print(f"New max sequence length is {dataset.xs.shape[1]}")

Average sequence length: 66+-38
Old max sequence length is 274
New max sequence length is 143


In [53]:
dataset_train, dataset_test = split_data_along_sessiondim(dataset, list_test_sessions=[3, 6, 9])

print(f"Number of training sessions: {dataset_train.xs.shape[0]}")
print(f"Number of test sessions: {dataset_test.xs.shape[0]}")

Number of training sessions: 567
Number of test sessions: 189


# SPICE

## SPICE Setup

Thoughts on SPICE architecture:

1. Reward and Effort sensitivity -> Separate values for reward and effort
2. Include effort levels for both tasks
3. Item-Space -> Task-Space -> Enables tracking task-individual values (e.g. some people might find easier to say color and others prefer form) 
4. Action-Space -> Repeat/Switch or specific Task-Selection?
5. Include separate module for repeat/switch bias -> May be unncessary because could be encoded already in effort modules
6. Include previous response time as input to repeat/switch bias
7. Account for length of experiment as cognitive control becomes more costly over time i.e. blocks ("We predict that the indifference point will increase as the experiment progresses")
8. Check whether state-less implementation makes sense -> Everything is given in cues (rewards for each task; effort levels could be directly predicted using last n actions) vs state-based dynamical model (evolution of effort/reward values over trials)

On repeat/switch bias:

"control becomes increasingly costly as it is exerted for a longer period of time" - Braun (2018)
-> Switching to keep optimal reward strength cannot be done over an extended period
-> Repeat/Switch ratio is probabily very unbalanced while Task-Ratio should be more or less counterbalanced -> **Benefit of acting on Task-Space instead of Repeat/Switch-Space**

Now we are going to define the configuration for SPICE with a `SpiceConfig` object.

The `SpiceConfig` takes as arguments 
1. `library_setup (dict)`: Defining the variable names of each module.
2. `memory_state (dict)`: Defining the memory state variables and their initial values.
3. `states_in_logit (list)`: Defining which of the memory state variables are used later for the logit computation. This is necessary for some background processes.  

In [9]:
spice_config = SpiceConfig(
    library_setup={
        'reward_repeat': (
            'dreward_tasks', 
            # 'dreward_trials_repeat',
            # 'dreward_trials_switch',
            ),
        'reward_switch': (
            'dreward_tasks',
            # 'dreward_trials_repeat',
            # 'dreward_trials_switch', 
            ),
        'task_repeat': (
            'repeat',
            ),
        'task_switch': (
            'repeat',
            ),
        'fatigue_repeat': (
            'block',
            ),
        'fatigue_switch': (
            'block',
            ),
    },
    memory_state={
        'value_reward': 0,
        'value_control': 0,
        'value_fatigue': 0,
    },
)

And now we are going to define the SPICE model which is a child of the `BaseModel` and `torch.nn.Module` class and takes as required arguments:
1. `spice_config (SpiceConfig)`: previously defined SpiceConfig object
2. `n_actions (int)`: number of possible actions in your dataset (including non-displayed ones if applicable).
3. `n_participants (int)`: number of participants in your dataset.

As usual for a `torch.nn.Module` we have to define at least the `__init__` method and the `forward` method.
The `forward` method gets called when computing a forward pass through the model and takes as inputs `(inputs (SpiceDataset.xs), prev_state (dict, default: None), batch_first (bool, default: False))` and returns `(logits (torch.Tensor, shape: (n_participants*n_blocks*n_experiments, timesteps, n_actions)), updated_state (dict))`. Two necessary method calls inside the forward pass are:
1. `self.init_forward_pass(inputs, prev_state, batch_first) -> SpiceSignals`: returns a `SpiceSignals` object which carries all relevant information already processed.
2. `self.post_forward_pass(SpiceSignals, batch_first) -> SpiceSignals`: does some re-arranging of the logits to adhere to `batch_first`.

In [54]:
class SpiceModel(BaseModel):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        
        dropout = 0.1
        self.n_blocks = 12
        
        self.participant_embedding = self.setup_embedding(num_embeddings=self.n_participants, embedding_size=self.embedding_size, dropout=dropout)
        
        self.setup_module(key_module='reward_repeat', input_size=1 + self.embedding_size, dropout=dropout, include_state=False)
        self.setup_module(key_module='reward_switch', input_size=1 + self.embedding_size, dropout=dropout, include_state=False)
        self.setup_module(key_module='task_repeat', input_size=1 + self.embedding_size, dropout=dropout)
        self.setup_module(key_module='task_switch', input_size=1 + self.embedding_size, dropout=dropout)
        self.setup_module(key_module='fatigue_repeat', input_size=1 + self.embedding_size, dropout=dropout, include_state=False)        
        self.setup_module(key_module='fatigue_switch', input_size=1 + self.embedding_size, dropout=dropout, include_state=False)        
        
    def forward(self, inputs, prev_state=None, batch_first=True):
        
        spice_signals = self.init_forward_pass(inputs, prev_state, batch_first)
        
        dreward_tasks_current = -spice_signals.additional_inputs[..., 0].unsqueeze(-1).expand_as(spice_signals.actions)
        dreward_tasks_other = spice_signals.additional_inputs[..., 0].unsqueeze(-1).expand_as(spice_signals.actions)
        dreward_trials_current = spice_signals.additional_inputs[..., 1].unsqueeze(-1).expand_as(spice_signals.actions)
        dreward_trials_other = spice_signals.additional_inputs[..., 2].unsqueeze(-1).expand_as(spice_signals.actions)
        # rt = spice_signals.additional_inputs[..., 3].unsqueeze(-1).expand_as(spice_signals.actions)
        blocks = spice_signals.blocks.unsqueeze(0).unsqueeze(-1).expand_as(spice_signals.actions[0]) / self.n_blocks
        
        repeat = torch.zeros_like(spice_signals.actions)
        repeat += spice_signals.actions[..., :1]
        switch = torch.zeros_like(spice_signals.actions)
        switch += spice_signals.actions[..., 1:2]
        
        repeat_mask = torch.zeros_like(spice_signals.actions[0])
        repeat_mask[..., 0] = 1
        switch_mask = torch.zeros_like(spice_signals.actions[0])
        switch_mask[..., 1] = 1
        
        participant_embedding = self.participant_embedding(spice_signals.participant_ids)
        
        for trial in spice_signals.trials:
            
            # modules for reward perception
            value_reward_repeat = self.call_module(
                key_module='reward_repeat',
                # key_state='value_reward',
                action_mask=repeat_mask,
                inputs=dreward_tasks_current[trial],
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding,
            )
            value_reward_switch = self.call_module(
                key_module='reward_switch',
                # key_state='value_reward',
                action_mask=switch_mask,
                inputs=dreward_tasks_other[trial],
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding,
            )
            
            # modules to update switching costs
            self.call_module(
                key_module='task_repeat',
                key_state='value_control',
                action_mask=repeat_mask,
                inputs=repeat[trial],
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding,
            )
            self.call_module(
                key_module='task_switch',
                key_state='value_control',
                action_mask=switch_mask,
                inputs=repeat[trial],
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding,
            )
            
            # module to update fatigue value based on current block number
            value_fatigue_repeat = self.call_module(
                key_module='fatigue_repeat',
                # key_state='value_fatigue',
                action_mask=repeat_mask,
                inputs=blocks,
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding,
            )
            value_fatigue_switch = self.call_module(
                key_module='fatigue_switch',
                # key_state='value_fatigue',
                action_mask=switch_mask,
                inputs=blocks,
                participant_index=spice_signals.participant_ids,
                participant_embedding=participant_embedding,
            )
            
            # compute logits
            spice_signals.logits[trial] = (
                value_reward_repeat+value_reward_switch
                + self.state['value_control'] 
                + value_fatigue_repeat+value_fatigue_switch
                )
        
        spice_signals = self.post_forward_pass(spice_signals, batch_first)
        return spice_signals.logits, self.state


Let's setup now the `SpiceEstimator` object and fit it to the data!

In [55]:
path_spice = '../params/braun2018/spice_braun2018.pkl'

estimator = SpiceEstimator(
        # model paramaeters
        spice_class=SpiceModel,
        spice_config=spice_config,
        n_actions=2,
        n_participants=n_participants,
        n_experiments=1,
        n_reward_features=0,
        
        epochs=1000,
        warmup_steps=200,
        ensemble_size=10,
        
        sindy_weight=0.1,
        sindy_alpha=0.0001,
        sindy_pruning_frequency=100,
        sindy_threshold_pruning=0.01,
        sindy_ensemble_pruning=0.05,
        sindy_library_polynomial_degree=2,
        
        sindy_refit=False,
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
        verbose=True,
        save_path_spice=path_spice,
    )

In [56]:
if train_spice:
    estimator.fit(dataset_train.xs, dataset_train.ys, dataset_test.xs, dataset_test.ys)

In [57]:
estimator.load_spice(path_spice)

# BENCHMARKING

### Expected Value of Control Model (Shenhav et al., 2013)

In [59]:
evc = ExpectedValueControl(n_participants=n_participants)
path_evc = path_spice.replace('spice_', 'evc_')

In [60]:
if train_evc:
    optimizer_evc = torch.optim.Adam(evc.parameters(), lr=0.01)
    epochs = 1000

    evc = training(
        model=evc,
        optimizer=optimizer_evc,
        dataset_train=dataset_train,
        dataset_test=dataset_test,
        epochs=epochs,
        device=torch.device('cpu'),
    )

    torch.save(evc.state_dict(), path_evc)

In [61]:
evc.load_state_dict(torch.load(path_evc, map_location='cpu'))

<All keys matched successfully>

### GRU Model

In [62]:
gru = GRU(n_actions, additional_inputs=4, n_reward_features=0)
path_gru = path_spice.replace('spice_', 'gru_')

In [63]:
if train_gru:
    epochs = 1000
    optimizer_gru = torch.optim.Adam(gru.parameters(), lr=0.01)

    gru = training(
        model=gru.to(torch.device('cuda' if torch.cuda.is_available() else 'cpu')),
        optimizer=optimizer_gru,
        dataset_train=dataset_train,
        dataset_test=dataset_test,
        epochs=epochs,
        ).to(torch.device('cpu'))

    torch.save(gru.state_dict(), path_gru)

Epoch 1/1000: L(Train): 0.7022228240966797; L(Test): 0.6754243969917297
Epoch 2/1000: L(Train): 0.6724429726600647; L(Test): 0.6668117046356201
Epoch 3/1000: L(Train): 0.659993052482605; L(Test): 0.6607396602630615
Epoch 4/1000: L(Train): 0.6520838737487793; L(Test): 0.6507664918899536
Epoch 5/1000: L(Train): 0.6422415375709534; L(Test): 0.6377213597297668
Epoch 6/1000: L(Train): 0.6302168369293213; L(Test): 0.624111533164978
Epoch 7/1000: L(Train): 0.6184144616127014; L(Test): 0.6117025017738342
Epoch 8/1000: L(Train): 0.6076127886772156; L(Test): 0.6004595160484314
Epoch 9/1000: L(Train): 0.5979776978492737; L(Test): 0.5886890292167664
Epoch 10/1000: L(Train): 0.5867869257926941; L(Test): 0.5754860639572144
Epoch 11/1000: L(Train): 0.5726652145385742; L(Test): 0.5621552467346191
Epoch 12/1000: L(Train): 0.5577712059020996; L(Test): 0.5503264665603638
Epoch 13/1000: L(Train): 0.5460419654846191; L(Test): 0.5382659435272217
Epoch 14/1000: L(Train): 0.5335555076599121; L(Test): 0.524016

In [64]:
gru.load_state_dict(torch.load(path_gru, map_location='cpu'))
# gru.to(torch.device('cpu')).eval()

<All keys matched successfully>

# ANALYSIS

In [65]:
from weinhardt2026.analysis.analysis_model_evaluation import analysis_model_evaluation

In [66]:
print("Model evaluation on training data (for AIC and BIC): ")
analysis_model_evaluation(
    dataset=dataset_train,
    spice_model=estimator,
    benchmark_model=evc,
    gru_model=gru,
)

Model evaluation on training data (for AIC and BIC): 
Computing choice probabilities with benchmark model...
Computing choice probabilities with GRU model...
Computing choice probabilities with SPICE model...


,Trial Lik.,(std),n_parameters,(std),NLL,(std),AIC,BIC
Benchmark,0.570510,0.155808,5.000000,0.000000,0.561225,0.235656,1.276921,1.434114
GRU,0.688373,0.152831,1778.000000,0.000000,0.373424,0.220744,55.676590,111.574440
SPICE-RNN,0.662736,0.148140,88652.000000,0.000000,0.411378,0.213409,2739.649170,5526.744141
SPICE,0.651254,0.148986,13.968255,2.451447,0.428856,0.223454,1.289249,1.729760


In [67]:
print("Model evaluation on held-out data (for average trial likelihood and NLL): ")
analysis_model_evaluation(
    dataset=dataset_test,
    spice_model=estimator,
    benchmark_model=evc,
    gru_model=gru,
)

Model evaluation on held-out data (for average trial likelihood and NLL): 
Computing choice probabilities with benchmark model...
Computing choice probabilities with GRU model...
Computing choice probabilities with SPICE model...


,Trial Lik.,(std),n_parameters,(std),NLL,(std),AIC,BIC
Benchmark,0.546267,0.159641,5.000000,0.000000,0.604647,0.263329,1.360615,1.516752
GRU,0.652959,0.152215,1778.000000,0.000000,0.426242,0.225513,54.662266,110.184250
SPICE-RNN,0.646598,0.147069,88652.000000,0.000000,0.436030,0.216039,2683.855469,5452.210938
SPICE,0.634173,0.151276,13.968255,2.455789,0.455433,0.236168,1.333604,1.771229
